# S&P 500: one-date feature test → Kafka → S3
Run sections 1–4 first. They use your local CSV only. Sections 5–7 are optional external operations.
The notebook is self-contained: no separate Python script is needed.

At Wednesday t, training contains X(t−1) paired with return t−1→t (plus earlier examples); inference contains X(t). Holiday Wednesdays use the preceding NYSE session. Individual missing quotes are never carried forward.
All inputs must be end-of-day data. The targets are close-to-close research labels, not executable post-signal returns.
The 472-stock survivor-universe limitation remains.


In [ ]:
%pip install pandas numpy pandas_market_calendars boto3 'kafka-python==2.2.16'


## 1. Configuration
Change TEST_DATE to a Wednesday covered by your CSV. 2025-06-18 is an example. Use a new topic for these feature messages rather than mixing them with old demo messages. Create that topic on your existing broker before publishing. Bucket names are globally unique; change BUCKET if needed.


In [ ]:
import os
from pathlib import Path
import json
import pandas as pd

CSV_PATH = Path("SP500_Historical_Data.csv")
TEST_DATE = "2025-06-18"  # assumes the session close has completed
OUTPUT_DIR = Path("feature_test") / TEST_DATE
AWS_PROFILE = "admin"
AWS_REGION = "ap-southeast-2"
BUCKET = "s3-stock-market-project-ashwin"  # create if absent
TOPIC = "sp500_features_v1"
BOOTSTRAP_SERVERS = os.environ.get("KAFKA_BOOTSTRAP_SERVERS", "localhost:9092").split(",")
CONSUMER_GROUP = "sp500-features-s3-v1"
FEATURE_VERSION = "close_features_v1"
DATA_VERSION = "kaggle_2026_02"  # change when you replace/revise the CSV
PUBLISH = False  # set True ONLY after reviewing the local results


## 2. Feature implementation
Close-only baseline: weekly returns, 12–1 momentum, daily volatility, moving-average distance, Bollinger position, simple RSI and closing-high distance. Other OHLCV columns are not used. About 52 weeks of history are required. Missing session prices remain missing; incomplete feature rows are excluded. This function truncates input at the requested date before computing features.


In [ ]:
#!/usr/bin/env python3
"""Close-based weekly features for SP500_Historical_Data.csv.

Install: pip install pandas numpy pandas_market_calendars
Run: python build_sp500_features.py --input SP500_Historical_Data.csv
Optional: --as-of 2026-02-18 --output features_20260218

Assumes complete end-of-day data through --as-of (default: max CSV date).
Wednesday anchors use the last NYSE session on/before Wednesday. A missing
stock quote on that session stays missing; it is NOT a holiday fallback.
Training is X(t) -> adjusted-close return from t to t+1; inference is X(latest).
These are research labels, NOT executable returns after observing close t.
For an executable backtest, separately align entry/exit prices after signals.
No survivorship correction or sector neutralization is performed.
Outputs are CSV plus metadata.json. Pass ONLY metadata['feature_columns'] to ML.
"""
import argparse
import json
from pathlib import Path

import numpy as np
import pandas as pd
import pandas_market_calendars as mcal


def build(input_path, output_path, as_of=None):
    df = pd.read_csv(input_path, usecols=['Ticker', 'Date', 'Adj Close'])
    if df.empty or df[['Ticker', 'Date']].isna().any().any():
        raise ValueError('Empty data or missing ticker/date.')
    df['Ticker'] = df['Ticker'].astype(str).str.strip()
    df['Date'] = pd.to_datetime(df['Date'], errors='raise').dt.normalize()
    if (df['Ticker'] == '').any() or df.duplicated(['Ticker', 'Date']).any():
        raise ValueError('Blank ticker or duplicate ticker/date: fix input first.')
    df['Adj Close'] = pd.to_numeric(df['Adj Close'], errors='raise')
    invalid = df['Adj Close'].isna() | ~np.isfinite(df['Adj Close']) | (df['Adj Close'] <= 0)
    if invalid.any():
        raise ValueError(f'{invalid.sum()} invalid adjusted prices: investigate first.')
    source_max = df['Date'].max()
    cutoff = pd.Timestamp(as_of).normalize() if as_of else source_max
    if cutoff < df['Date'].min():
        raise ValueError('As-of precedes the data.')
    df = df.loc[df['Date'] <= cutoff].copy()
    # Include a buffer so an early holiday anchor can find its preceding session.
    sessions = mcal.get_calendar('NYSE').valid_days(
        start_date=df['Date'].min() - pd.Timedelta(days=10), end_date=cutoff
    ).tz_localize(None)
    if not df['Date'].isin(sessions).all():
        raise ValueError('Input contains dates outside the NYSE session calendar.')
    anchors = pd.DataFrame({'week_date': pd.date_range(df['Date'].min(), cutoff, freq='W-WED')})
    if anchors.empty:
        raise ValueError('No completed Wednesday anchors in this range.')
    calendar = pd.merge_asof(
        anchors, pd.DataFrame({'price_date': sessions}),
        left_on='week_date', right_on='price_date', direction='backward'
    )
    if calendar['price_date'].iloc[-1] > df['Date'].max():
        raise ValueError('Latest required snapshot is after available data; update CSV or reduce --as-of.')
    calendar['label_end_week'] = calendar['week_date'] + pd.Timedelta(days=7)
    calendar['label_end_date'] = calendar['price_date'].shift(-1)
    feature_columns = [
        'return_1w', 'return_4w', 'return_13w', 'return_26w', 'momentum_12_1',
        'volatility_20d', 'volatility_60d', 'price_to_sma20', 'price_to_sma60',
        'bollinger_z20', 'rsi14_simple', 'close_to_high252',
    ]
    parts = []
    missing_sessions = 0
    for ticker, group in df.groupby('Ticker', sort=True):
        # Reindex to sessions: rolling windows cannot silently skip missing days.
        grid = sessions[(sessions >= group['Date'].min()) & (sessions <= cutoff)]
        p = group.set_index('Date')['Adj Close'].reindex(grid)
        missing_sessions += int(p.isna().sum())
        r = p.pct_change(fill_method=None)
        daily = pd.DataFrame({'price_date': grid, 'adj_close': p.to_numpy()})
        mean20, std20 = p.rolling(20).mean(), p.rolling(20).std()
        daily['volatility_20d'] = (r.rolling(20).std() * np.sqrt(252)).to_numpy()
        daily['volatility_60d'] = (r.rolling(60).std() * np.sqrt(252)).to_numpy()
        daily['price_to_sma20'] = (p / mean20 - 1).to_numpy()
        daily['price_to_sma60'] = (p / p.rolling(60).mean() - 1).to_numpy()
        z = (p - mean20) / std20.replace(0, np.nan)
        daily['bollinger_z20'] = z.mask(std20.eq(0), 0).to_numpy()
        delta = p.diff()
        gain = delta.clip(lower=0).rolling(14).mean()
        loss = (-delta.clip(upper=0)).rolling(14).mean()
        # Simple rolling RSI, deliberately not Wilder-smoothed RSI.
        rsi = 100 * gain / (gain + loss)
        daily['rsi14_simple'] = rsi.mask((gain + loss).eq(0), 50).to_numpy()
        daily['close_to_high252'] = (p / p.rolling(252).max() - 1).to_numpy()
        weekly = calendar.merge(daily, on='price_date', how='left', validate='one_to_one')
        weekly.insert(0, 'Ticker', ticker)
        for weeks in (1, 4, 13, 26):
            weekly[f'return_{weeks}w'] = weekly['adj_close'] / weekly['adj_close'].shift(weeks) - 1
        # Approximate 12-minus-1-month momentum using exact 52/4 weekly anchors.
        weekly['momentum_12_1'] = weekly['adj_close'].shift(4) / weekly['adj_close'].shift(52) - 1
        weekly['target_return_1w'] = weekly['adj_close'].shift(-1) / weekly['adj_close'] - 1
        parts.append(weekly)
    panel = pd.concat(parts, ignore_index=True).sort_values(['week_date', 'Ticker'])
    panel[feature_columns] = panel[feature_columns].replace([np.inf, -np.inf], np.nan)
    eligible = panel[feature_columns].notna().all(axis=1) & panel['adj_close'].notna()
    latest = calendar['week_date'].iloc[-1]
    training = panel.loc[eligible & panel['target_return_1w'].notna() &
                         panel['label_end_week'].le(latest)].copy()
    # Equal outcomes get equal relevance. Categories can be empty in small groups.
    pct = training.groupby('week_date')['target_return_1w'].rank(method='average', pct=True)
    training['target_relevance'] = np.minimum(np.ceil(pct * 10) - 1, 9).astype(int)
    inference_columns = ['Ticker', 'week_date', 'price_date', 'adj_close'] + feature_columns
    inference = panel.loc[eligible & panel['week_date'].eq(latest), inference_columns].copy()
    if inference.empty:
        raise ValueError('No eligible inference rows. Need about 52 weeks of history and complete windows.')
    out = Path(output_path)
    out.mkdir(parents=True, exist_ok=True)
    panel.to_csv(out / 'weekly_panel.csv', index=False)
    training.to_csv(out / 'training.csv', index=False)
    inference.to_csv(out / 'inference.csv', index=False)
    calendar.to_csv(out / 'snapshot_calendar.csv', index=False)
    metadata = {
        'input': str(input_path), 'source_max_date': str(source_max.date()),
        'as_of': str(cutoff.date()), 'inference_week': str(latest.date()),
        'inference_price_date': str(calendar['price_date'].iloc[-1].date()),
        'feature_columns': feature_columns, 'regression_target': 'target_return_1w',
        'ranking_target': 'target_relevance', 'ranking_group': 'week_date',
        'training_rows': len(training), 'inference_rows': len(inference),
        'missing_ticker_sessions_including_after_last_quote': missing_sessions,
        'latest_ineligible_tickers': panel.loc[panel['week_date'].eq(latest) & ~eligible, 'Ticker'].tolist(),
        'target_definition': 'X(t) -> AdjClose(next weekly snapshot)/AdjClose(t)-1; no second feature shift',
        'warning': 'Research close-to-close labels; not post-signal execution returns. Survivor universe remains biased.',
        'versions': {'pandas': pd.__version__, 'numpy': np.__version__, 'pandas_market_calendars': mcal.__version__},
    }
    (out / 'metadata.json').write_text(json.dumps(metadata, indent=2) + '\n')
    print(json.dumps(metadata, indent=2))
    return panel, training, inference, metadata




## 3. Build and inspect one date


In [ ]:
assert CSV_PATH.exists(), f"File not found: {CSV_PATH.resolve()}"
assert pd.Timestamp(TEST_DATE).weekday() == 2, "Choose a Wednesday anchor."
weekly_panel, training, inference, metadata = build(CSV_PATH, OUTPUT_DIR, TEST_DATE)
FEATURE_COLUMNS = metadata["feature_columns"]
assert inference["week_date"].eq(pd.Timestamp(TEST_DATE)).all()
assert training["label_end_week"].le(pd.Timestamp(TEST_DATE)).all()
assert training["week_date"].lt(pd.Timestamp(TEST_DATE)).all()
assert inference[FEATURE_COLUMNS].notna().all().all()
assert not inference.duplicated(["Ticker", "week_date"]).any()

display(inference.head())
display(training.tail())
print("Training rows:", len(training), "Inference stocks:", len(inference))
print("Actual snapshot session:", metadata["inference_price_date"])
print("Ineligible stocks:", metadata["latest_ineligible_tickers"])

# Inspect the newest labelled cohort: its features are one week older.
latest_training_week = training["week_date"].max()
display(training.loc[training["week_date"].eq(latest_training_week),
    ["Ticker", "week_date", "price_date", "label_end_week",
     "label_end_date", "target_return_1w"]].head())

# These are the inputs for later model training/inference.
X_train = training[FEATURE_COLUMNS]
y_train = training["target_return_1w"]
X_inference = inference[FEATURE_COLUMNS]
# For LightGBM ranking instead: target_relevance; group by week_date.
# Do not include dates, adjusted price, targets or label dates in X.


## 4. Prepare JSON messages (local only)
One message per stock per snapshot. Only inference features are published—never forward-return labels. A data version distinguishes revised datasets. The same stock/date/version writes the same S3 key, so reruns replace that snapshot rather than duplicating it.


In [ ]:
def make_messages(snapshot):
    records = snapshot.copy()
    records["week_date"] = pd.to_datetime(records["week_date"]).dt.strftime("%Y-%m-%d")
    records["price_date"] = pd.to_datetime(records["price_date"]).dt.strftime("%Y-%m-%d")
    rows = json.loads(records.to_json(orient="records", double_precision=15))
    return [
        {"feature_version": FEATURE_VERSION, "data_version": DATA_VERSION, **row}
        for row in rows
    ]

messages = make_messages(inference)
assert messages
print("Messages ready:", len(messages))
print(json.dumps(messages[0], indent=2, allow_nan=False))


## 5. Publish this snapshot to Kafka
Leave PUBLISH=False for the local test. When ready, ensure the broker/topic is available, set PUBLISH=True and run this cell. It sends each stock iteratively and waits for broker acknowledgement.
The consumer can run afterwards while messages remain retained in Kafka. For ongoing use, run the consumer in a separate process/kernel.


In [ ]:
from kafka import KafkaProducer

def publish_messages(records):
    producer = KafkaProducer(
        bootstrap_servers=BOOTSTRAP_SERVERS,
        key_serializer=lambda k: k.encode("utf-8"),
        value_serializer=lambda v: json.dumps(v, allow_nan=False).encode("utf-8"),
        acks="all",
        retries=3,
        max_in_flight_requests_per_connection=1,
    )
    sent = 0
    try:
        for record in records:
            key = "|".join([record["feature_version"], record["data_version"],
                            record["week_date"], record["Ticker"]])
            producer.send(TOPIC, key=key, value=record).get(timeout=60)
            sent += 1
        producer.flush(timeout=60)
    finally:
        producer.close(timeout=10)
    return sent

if PUBLISH:
    print("Acknowledged messages:", publish_messages(messages))
else:
    print("Dry run: no messages sent. Review results, then set PUBLISH=True.")


## 6. Kafka subscriber → S3
First authenticate in a terminal: `aws sso login --profile admin`.
Create BUCKET in ap-southeast-2 if it does not exist. This code does not create it.

This cell defines the consumer; the following cell starts it. Each successful S3 write is followed by an explicit commit of that partition's next offset. Failed uploads stop processing without acknowledging that message. Replaying it writes the same key (at-least-once delivery with idempotent snapshot keys, not a cross-service transaction).

A new consumer group reads retained messages from earliest; an existing group resumes at committed offsets. Do not reset offsets just to retry a failed upload.
For the notebook test, exit after 15 seconds without messages. Set idle_timeout_ms=None for a continuous subscriber.

Small per-stock JSON objects are convenient for this test. Later compact them to date-partitioned Parquet for Athena. S3 writes alone do not create an Athena table.


In [ ]:
import boto3
from kafka import KafkaConsumer
from kafka.structs import TopicPartition, OffsetAndMetadata
from urllib.parse import quote

def s3_key(record):
    required = ["feature_version", "data_version", "week_date", "price_date", "Ticker"]
    if not all(record.get(k) for k in required):
        raise ValueError("Invalid feature message: missing identifiers.")
    return (
        f"features/feature_version={quote(str(record['feature_version']), safe='')}/"
        f"data_version={quote(str(record['data_version']), safe='')}/"
        f"week_date={record['week_date']}/"
        f"{quote(str(record['Ticker']), safe='')}.json"
    )

def store_message(s3_client, consumer, message):
    record = message.value
    if record.get("feature_version") != FEATURE_VERSION:
        raise ValueError("Unexpected feature schema; message was not committed.")
    if not all(k in record for k in FEATURE_COLUMNS):
        raise ValueError("Missing feature columns; message was not committed.")
    key = s3_key(record)
    s3_client.put_object(
        Bucket=BUCKET, Key=key,
        Body=(json.dumps(record, allow_nan=False) + "\n").encode("utf-8"),
        ContentType="application/json",
    )
    partition = TopicPartition(message.topic, message.partition)
    consumer.commit({partition: OffsetAndMetadata(message.offset + 1, "", -1)})
    return key

def consume_to_s3(idle_timeout_ms=15000):
    session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)
    s3_client = session.client("s3")
    s3_client.head_bucket(Bucket=BUCKET)  # fail early if absent/inaccessible
    config = dict(
        bootstrap_servers=BOOTSTRAP_SERVERS,
        group_id=CONSUMER_GROUP,
        auto_offset_reset="earliest",
        enable_auto_commit=False,
        max_poll_records=1,
        value_deserializer=lambda b: json.loads(b.decode("utf-8")),
    )
    if idle_timeout_ms is not None:
        config["consumer_timeout_ms"] = idle_timeout_ms
    consumer = KafkaConsumer(TOPIC, **config)
    uploaded = 0
    try:
        for message in consumer:
            key = store_message(s3_client, consumer, message)
            uploaded += 1
            print(f"Uploaded s3://{BUCKET}/{key}")
    finally:
        consumer.close()
    print("Uploaded in this invocation:", uploaded)
    return uploaded


In [ ]:
# Run after publishing. This cell writes to S3.
consume_to_s3()

# Later, run continuously in a separate process/kernel:
# consume_to_s3(idle_timeout_ms=None)


## 7. Optional S3 read-back
Uncomment after consuming the snapshot. This reads the first stock's stored features.


In [ ]:
# session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)
# s3 = session.client("s3")
# saved = s3.get_object(Bucket=BUCKET, Key=s3_key(messages[0]))
# display(json.loads(saved["Body"].read()))


## Later: iterate over Wednesday snapshots
The local test already creates training.csv and inference.csv. Publish each date's inference snapshot once; do not resend the entire expanding training dataset every week.
The simple loop below recomputes history for clarity. For a full historical backfill, compute the panel once and iterate its date groups to avoid repeated CSV reads.
When future prices arrive, training labels are constructed by joining consecutive stored snapshots. Never attach a future label to an inference-time message.


In [ ]:
# Uncomment only after the one-date Kafka/S3 test succeeds.
# for date in pd.date_range("2025-06-18", "2025-07-02", freq="W-WED"):
#     date_string = date.strftime("%Y-%m-%d")
#     _, _, snapshot, _ = build(CSV_PATH, Path("feature_runs") / date_string, date_string)
#     print(date_string, publish_messages(make_messages(snapshot)))


References: [Exchange calendar](https://pandas-market-calendars.readthedocs.io/en/latest/usage.html), [Kafka consumer](https://kafka-python.readthedocs.io/en/2.2.16/apidoc/KafkaConsumer.html), [S3 put_object](https://docs.aws.amazon.com/boto3/latest/reference/services/s3/client/put_object.html).
